# Ordering

Adapted from [ib_async ordering](https://ib-api-reloaded.github.io/ib_async/notebooks.html).

**WARNING: This notebook places live orders. Use a paper trading account.**

In [ ]:
import os, threading, time
from dotenv import load_dotenv
from ibx import EWrapper, EClient, Contract, Order

load_dotenv()
USERNAME = os.environ["IB_USERNAME"]
PASSWORD = os.environ["IB_PASSWORD"]

In [ ]:
class App(EWrapper):
    def __init__(self):
        super().__init__()
        self.next_id = None
        self.order_statuses = []  # (order_id, status, filled, remaining)
        self.positions_list = []
        self.connected = threading.Event()

    def next_valid_id(self, order_id):
        self.next_id = order_id
        self.connected.set()

    def order_status(self, order_id, status, filled, remaining,
                     avg_fill_price, perm_id, parent_id,
                     last_fill_price, client_id, why_held, mkt_cap_price):
        self.order_statuses.append((order_id, status, filled, remaining, avg_fill_price))
        print(f"  OrderStatus: id={order_id} status={status} filled={filled} remaining={remaining} avgPrice={avg_fill_price:.2f}")

    def position(self, account, contract, pos, avg_cost):
        self.positions_list.append((contract.symbol, pos, avg_cost))

    def position_end(self):
        pass

    def error(self, req_id, error_code, error_string, advanced_order_reject_json=""):
        if error_code not in (2104, 2106, 2158):
            print(f"Error {error_code}: {error_string}")

In [ ]:
app = App()
client = EClient(app)
client.connect(username=USERNAME, password=PASSWORD, paper=True)

thread = threading.Thread(target=client.run, daemon=True)
thread.start()
app.connected.wait(timeout=10)
print(f"Connected: {client.is_connected()}")
print(f"Next order ID: {app.next_id}")

### Subscribe to AAPL market data

We need a market data subscription before placing orders (for instrument registration):

In [ ]:
aapl = Contract(con_id=265598, symbol="AAPL")
client.req_mkt_data(1, aapl)
time.sleep(2)

### Place a limit order (unrealistic price, won't fill)

Buy 1 share of AAPL at $1.00 — should be Submitted but never filled:

In [ ]:
app.order_statuses.clear()

order_id = app.next_id
limit_order = Order(order_id=order_id, action="BUY", total_quantity=1,
                    order_type="LMT", lmt_price=1.00)

print(f"Placing limit order {order_id}: BUY 1 AAPL @ $1.00")
client.place_order(order_id, aapl, limit_order)
time.sleep(2)

print(f"\nOrder statuses received: {len(app.order_statuses)}")

### Cancel the order

In [ ]:
print(f"Cancelling order {order_id}")
client.cancel_order(order_id)
time.sleep(2)

### Place a market order (will fill immediately)

Buy 1 share at market:

In [ ]:
app.order_statuses.clear()

mkt_order_id = order_id + 1
mkt_order = Order(order_id=mkt_order_id, action="BUY", total_quantity=1,
                  order_type="MKT")

print(f"Placing market order {mkt_order_id}: BUY 1 AAPL @ MKT")
client.place_order(mkt_order_id, aapl, mkt_order)
time.sleep(3)

print(f"\nOrder statuses received: {len(app.order_statuses)}")

### Check positions

In [ ]:
app.positions_list.clear()
client.req_positions()
time.sleep(1)

for symbol, pos, avg_cost in app.positions_list:
    print(f"  {symbol}: {pos:.0f} shares @ {avg_cost:.2f}")

### Sell back to flatten

In [ ]:
sell_order_id = mkt_order_id + 1
sell_order = Order(order_id=sell_order_id, action="SELL", total_quantity=1,
                   order_type="MKT")

print(f"Placing sell order {sell_order_id}: SELL 1 AAPL @ MKT")
client.place_order(sell_order_id, aapl, sell_order)
time.sleep(3)

In [ ]:
client.cancel_mkt_data(1)
client.disconnect()